In [14]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt 
from sklearn.model_selection import train_test_split 
from sklearn.ensemble import StackingClassifier
from sklearn.tree import DecisionTreeClassifier  
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC 
from sklearn.metrics import accuracy_score, classification_report
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

In [2]:
df = pd.read_csv('seattle-weather.csv')
df.head()

,date,precipitation,temp_max,temp_min,wind,weather
0,2012-01-01,0.0,12.8,5.0,4.7,drizzle
1,2012-01-02,10.9,10.6,2.8,4.5,rain
2,2012-01-03,0.8,11.7,7.2,2.3,rain
3,2012-01-04,20.3,12.2,5.6,4.7,rain
4,2012-01-05,1.3,8.9,2.8,6.1,rain


In [5]:
df.info()
df.duplicated().sum()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1461 entries, 0 to 1460
Data columns (total 6 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1461 non-null   object 
 1   precipitation  1461 non-null   float64
 2   temp_max       1461 non-null   float64
 3   temp_min       1461 non-null   float64
 4   wind           1461 non-null   float64
 5   weather        1461 non-null   object 
dtypes: float64(4), object(2)
memory usage: 68.6+ KB


,precipitation,temp_max,temp_min,wind
count,1461.000000,1461.000000,1461.000000,1461.000000
mean,3.029432,16.439083,8.234771,3.241136
std,6.680194,7.349758,5.023004,1.437825
min,0.000000,-1.600000,-7.100000,0.400000
25%,0.000000,10.600000,4.400000,2.200000
50%,0.000000,15.600000,8.300000,3.000000
75%,2.800000,22.200000,12.200000,4.000000
max,55.900000,35.600000,18.300000,9.500000


In [12]:
df.dtypes

precipitation    float64
temp_max         float64
temp_min         float64
wind             float64
weather           object
year               int32
month              int32
day                int32
dtype: object

In [8]:
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year
df['month'] = df['date'].dt.month
df['day'] = df['date'].dt.day

In [10]:
df.drop('date',axis=1,inplace=True)

In [11]:
df.head()

,precipitation,temp_max,temp_min,wind,weather,year,month,day
0,0.0,12.8,5.0,4.7,drizzle,2012,1,1
1,10.9,10.6,2.8,4.5,rain,2012,1,2
2,0.8,11.7,7.2,2.3,rain,2012,1,3
3,20.3,12.2,5.6,4.7,rain,2012,1,4
4,1.3,8.9,2.8,6.1,rain,2012,1,5


In [13]:
X = df.drop('weather',axis=1)
y = df['weather']

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [36]:
from sklearn.preprocessing import StandardScaler
numeric_columns = X_train.select_dtypes(include=['int64', 'float64']).columns

scaler = StandardScaler()

# Fit ONLY on training data
X_train[numeric_columns] = scaler.fit_transform(X_train[numeric_columns])

# Transform test data using same scaler
X_test[numeric_columns] = scaler.transform(X_test[numeric_columns])


In [16]:
le = LabelEncoder()
y_encoded = le.fit_transform(y)

In [37]:
base_learners = [
    ('dt',DecisionTreeClassifier(random_state=42)),
    ('svc',SVC(probability=True,kernel='rbf',random_state=42)),
    ('lr',LogisticRegression(max_iter=1000))
]

In [38]:
meta_learner = LogisticRegression(max_iter=1000)

In [39]:
stacking_clf = StackingClassifier(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5
)

In [40]:
stacking_clf.fit(X_train,y_train)

C:\Users\tanis\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
C:\Users\tanis\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_

StackingClassifier(cv=5,
                   estimators=[('dt', DecisionTreeClassifier(random_state=42)),
                               ('svc', SVC(probability=True, random_state=42)),
                               ('lr', LogisticRegression(max_iter=1000))],
                   final_estimator=LogisticRegression(max_iter=1000))

In [41]:
y_pred = stacking_clf.predict(X_test)

In [42]:
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.8156996587030717

In [43]:
from sklearn.ensemble import AdaBoostClassifier,GradientBoostingClassifier 
from xgboost import XGBClassifier

In [44]:
ada_model = AdaBoostClassifier(n_estimators=200, random_state=42)

In [45]:
ada_model.fit(X_train,y_train)

AdaBoostClassifier(n_estimators=200, random_state=42)

In [46]:
y_pred = ada_model.predict(X_test)

In [47]:
accuracy = accuracy_score(y_test,y_pred)

In [48]:
accuracy

0.6689419795221843

In [49]:
gb_model = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, random_state=42)
gb_model.fit(X_train,y_train)

GradientBoostingClassifier(random_state=42)

In [50]:
y_pred = gb_model.predict(X_test)

In [51]:
accuracy = accuracy_score(y_test,y_pred)
accuracy

0.8327645051194539

In [52]:
import joblib
joblib.dump(gb_model, "weather_model.pkl")


['weather_model.pkl']